# Data Cleaning: MCO vs MIA Analysis

Cleans filtered data and creates derived columns for analysis.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
print(f"pandas {pd.__version__}")

## Load Data

In [ ]:
# Load combined data from notebook 01
df = pd.read_csv('../data/processed/mco_mia_2004_2008.csv', low_memory=False)

print(f"Loaded {len(df):,} rows")
print(f"Years: {df['Year'].min()}-{df['Year'].max()}")
print(f"Columns: {len(df.columns)}")

display(df.head())

## Data Cleaning

In [ ]:
# Create flight date
df['FlightDate'] = pd.to_datetime(
    df[['Year', 'Month', 'DayofMonth']].rename(columns={'DayofMonth': 'Day'}),
    errors='coerce'
)

# On-time performance (within 15 minutes)
df['OnTime'] = (df['ArrDelay'].fillna(999) <= 15) & (df['Cancelled'] == 0)

# Delay categories
df['DelayCategory'] = pd.cut(
    df['ArrDelay'],
    bins=[-np.inf, 0, 15, 60, np.inf],
    labels=['Early', 'OnTime', 'Delayed', 'SeverelyDelayed']
)

# Time features
df['Quarter'] = df['Month'].apply(lambda m: (m-1)//3 + 1)
df['IsHolidaySeason'] = df['Month'].isin([6, 7, 8, 11, 12])

# Day names
day_map = {1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat', 7: 'Sun'}
df['DayName'] = df['DayOfWeek'].map(day_map)

# Airport indicators
df['IsMCO'] = (df['Origin'] == 'MCO') | (df['Dest'] == 'MCO')
df['IsMIA'] = (df['Origin'] == 'MIA') | (df['Dest'] == 'MIA')

print("Created derived columns")

## Data Summary

In [ ]:
print(f"Total flights: {len(df):,}")
print(f"Date range: {df['FlightDate'].min().date()} to {df['FlightDate'].max().date()}")
print(f"Cancellation rate: {(df['Cancelled'].sum()/len(df))*100:.2f}%")

# On-time performance (exclude cancelled flights)
non_cancelled = df[df['Cancelled'] == 0]
otp_rate = (non_cancelled['OnTime'].sum() / len(non_cancelled)) * 100
avg_delay = non_cancelled['ArrDelay'].mean()

print(f"On-time performance: {otp_rate:.2f}%")
print(f"Average delay: {avg_delay:.1f} minutes")

print("\nFlights by year:")
print(df['Year'].value_counts().sort_index())

## Airport Comparison

In [ ]:
# Departures by airport
mco_departures = len(df[df['Origin'] == 'MCO'])
mia_departures = len(df[df['Origin'] == 'MIA'])

print(f"MCO departures: {mco_departures:,}")
print(f"MIA departures: {mia_departures:,}")

# On-time performance by airport
mco_otp = df[df['Origin'] == 'MCO']['OnTime'].mean() * 100
mia_otp = df[df['Origin'] == 'MIA']['OnTime'].mean() * 100

print(f"\nMCO OTP: {mco_otp:.2f}%")
print(f"MIA OTP: {mia_otp:.2f}%")

## Save Cleaned Data

In [ ]:
output_path = '../data/processed/mco_mia_clean.csv'
df.to_csv(output_path, index=False)

print(f"Saved {len(df):,} rows to {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024**2:.1f} MB")

## Visualizations

In [ ]:
# Annual flight volume
fig, ax = plt.subplots(figsize=(10, 5))
df['Year'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Total Flights by Year', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Flights')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# MCO vs MIA departures over time
mco_yearly = df[df['Origin'] == 'MCO'].groupby('Year').size()
mia_yearly = df[df['Origin'] == 'MIA'].groupby('Year').size()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mco_yearly.index, mco_yearly.values, marker='o', label='MCO', linewidth=2)
ax.plot(mia_yearly.index, mia_yearly.values, marker='o', label='MIA', linewidth=2)
ax.set_title('Annual Departures: MCO vs MIA', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Departures')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Delay distribution
delay_counts = df['DelayCategory'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
delay_counts.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Flight Delay Distribution', fontsize=14)
ax.set_xlabel('Category')
ax.set_ylabel('Number of Flights')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

## Next Steps

Data is ready for:
- Detailed analysis
- Additional visualizations
- Power BI/Tableau dashboards